In [6]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from folium.plugins import MarkerCluster

In [7]:
ciudad_lineal = '2807915'

hortaleza = '2807916'

In [ ]:
df_poblacion = (
    pd.read_csv('/Users/david/Desktop/renta/65034.csv', sep = '\t',dtype = 'category', usecols = ['Secciones','Sexo','Edad','Periodo','Total'])
    .pipe(lambda df_:df_[df_['Periodo'] == '2024'])
    .pipe(lambda df_:df_[(df_['Sexo'] != 'Total') & (df_['Edad'] != 'Todas las edades')])
    .drop(columns = ['Periodo'])
    .pipe(lambda df_:df_[~df_['Secciones'].isna()])
    .assign(Secciones = lambda df_:df_['Secciones'].str.split(' ').str[0])
    .pipe(lambda df_:df_[df_['Secciones'].str.startswith('28')])
    .assign(Total = lambda df_:(df_['Total'].astype('str').str.replace('.','').fillna('0.0')))
    .assign(Total = lambda df_:np.select([df_['Total'] == ''],['0'],df_['Total']))
    .assign(Total = lambda df_:df_['Total'].fillna('0').replace({'nan':'0'}).astype('int'))
    .groupby(['Secciones',], as_index=False, observed=True)
    .agg({'Total':'mean'})
    .round(2)
    
)

df_poblacion

In [9]:
gdf = (
    gpd.read_file('/Users/david/Desktop/renta/seccionado_2024/SECC_CE_20240101.shp')
    .filter(['CUSEC','geometry'])
    .pipe(lambda df_:df_[df_['CUSEC'].str.startswith((hortaleza, ciudad_lineal))])
    .merge(df_poblacion, left_on = ['CUSEC'], right_on = ['Secciones'], how = 'left')
    .assign(density = lambda df_:df_['Total'] / df_['geometry'].area * 1000000)
    .filter(['CUSEC','density','geometry'])
    .to_crs('EPSG:4326')
)

In [ ]:
# Convert to JSON
geojson_data = gdf.to_json()

# Create a folium map centered on the data
m = folium.Map(location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10)

# Add Choropleth layer
choropleth = folium.Choropleth(
    geo_data=geojson_data,
    name="Choropleth",
    data=gdf,
    columns=["CUSEC", "density"],
    key_on="feature.properties.CUSEC",
    fill_color="YlGnBu",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Densidad en habitantes por kilómetro cuadrado",
).add_to(m)

# Add popups
folium.GeoJson(
    gdf,
    name="Popups",
    tooltip=folium.GeoJsonTooltip(
        fields=["CUSEC", "density"],  # Adjust fields to match your dataset
        aliases=["Region ID:", "Value:"],
        localize=True,
        sticky=True
    )
).add_to(m)

# Save map
m.save("../docs/map_density.html")